# 02 — Baseline model & submission

Same preprocessing for train and test. Predict every test row; export `submissions/baseline.csv` with columns `id` and `Accept`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))

from src.features import (
    ID_COL,
    TARGET_COL,
    build_preprocessor,
    load_train_test,
    prepare_features,
    split_features_target,
)

train, test = load_train_test(ROOT / "data")
X_train, y_train = split_features_target(train)
test_ids = test[ID_COL]
X_train = prepare_features(X_train)
X_test = prepare_features(test)

print(X_train.shape, X_test.shape, y_train.value_counts(normalize=True))

In [ ]:
model = Pipeline(
    steps=[
        ("prep", build_preprocessor(X_train)),
        (
            "clf",
            LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
        ),
    ]
)

scores = cross_val_score(model, X_train, y_train, cv=5, scoring="f1_macro", n_jobs=-1)
print(f"CV f1_macro: {scores.mean():.4f} (+/- {scores.std():.4f})")

model.fit(X_train, y_train)
preds = model.predict(X_test)

In [ ]:
sub_dir = ROOT / "submissions"
sub_dir.mkdir(parents=True, exist_ok=True)

submission = pd.DataFrame({ID_COL: test_ids, TARGET_COL: preds.astype(int)})
out_path = sub_dir / "baseline.csv"
submission.to_csv(out_path, index=False)

print(f"Saved {out_path}")
submission.head(), submission[TARGET_COL].value_counts()

Upload `submissions/baseline.csv` on Kaggle. When preprocessing stabilizes, use `python -m src.train` from the project root.